In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys
from typing import Dict, List

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from thermal_cooling.io import save_csv

In [ ]:
RAW_ROOT = ROOT / 'data' / 'raw' / 'source_data'
START_TIMES_PATH = ROOT / 'data' / 'raw' / 'session_start_times.csv'
PROCESSED_ROOT = ROOT / 'data' / 'processed' / 'aggregated_by_signal'
WINDOW_MINUTES = 70

CONDITION_MAP = {
    'no_wind': 'no_wind',
    'wind_low': 'wind_low',
    'wind_high': 'wind_high',
}

SIGNAL_GROUPS: Dict[str, List[str]] = {
    'T': [
        'T NECK',
        'T CHEST',
        'T ABDOMEN',
        'T DORSAL',
        'T HAND',
        'T SHIN',
        'T MICROCLIMATEFRONT',
        'T MICROCLIMATEBACK',
        'T ENVIRONMENT',
        'T CORE',
    ],
    'RH': [
        'RH NECK',
        'RH CHEST',
        'RH ABDOMEN',
        'RH DORSAL',
        'RH HAND',
        'RH SHIN',
        'RH MICROCLIMATEFRONT',
        'RH MICROCLIMATEBACK',
        'RH ENVIRONMENT',
        'RH CORE',
    ],
    'HR': ['HR'],
}

OUTPUT_COLUMNS = [
    'subject_id',
    'condition',
    'signal_group',
    'signal_name',
    'timestamp',
    'elapsed_seconds',
    'elapsed_minutes',
    'value',
]

In [ ]:
@dataclass(frozen=True)
class SessionInfo:
    subject_id: str
    condition: str
    start_time: pd.Timestamp

def normalize_condition(condition: str) -> str:
    key = condition.strip().lower()
    if key not in CONDITION_MAP:
        raise ValueError(
            f'Unsupported condition "{condition}". Allowed: {list(CONDITION_MAP)}'
        )
    return CONDITION_MAP[key]

def load_start_times(path: Path) -> List[SessionInfo]:
    if not path.exists():
        raise FileNotFoundError(
            f'Missing start-times file: {path}\n'
            'Create data/raw/session_start_times.csv with columns: '
            'subject_id,condition,start_time'
        )

    df = pd.read_csv(path)
    required = {'subject_id', 'condition', 'start_time'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'session_start_times.csv missing columns: {sorted(missing)}')

    df = df.copy()
    df['subject_id'] = df['subject_id'].astype(str).str.strip()
    df['condition'] = df['condition'].astype(str).str.strip()
    df['start_time'] = parse_datetime_series(df['start_time'])

    if df['start_time'].isna().any():
        bad = df.loc[df['start_time'].isna(), ['subject_id', 'condition', 'start_time']]
        raise ValueError(
            'Some start_time values could not be parsed. Accepted examples: '\
            '06/05/25 10:25:51 or 06/05/2025 10:25:51.\n'
            f'Problematic rows:\n{bad.to_string(index=False)}'
        )

    infos: List[SessionInfo] = []
    for row in df.itertuples(index=False):
        condition = normalize_condition(row.condition)
        infos.append(SessionInfo(row.subject_id, condition, row.start_time))
    return infos


def parse_datetime_series(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series, format='%d/%m/%y %H:%M:%S', errors='coerce')
    missing = parsed.isna()
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(
            series.loc[missing], format='%d/%m/%Y %H:%M:%S', errors='coerce'
        )
    missing = parsed.isna()
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(
            series.loc[missing], format='%d/%m/%y %H:%M', errors='coerce'
        )
    missing = parsed.isna()
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(
            series.loc[missing], format='%d/%m/%Y %H:%M', errors='coerce'
        )
    return parsed

In [ ]:
def find_signal_file(session_dir: Path, signal_group: str, expected_name: str) -> Path:
    folder = session_dir / signal_group
    if not folder.exists():
        raise FileNotFoundError(f'Missing folder: {folder}')

    expected_csv = folder / f'{expected_name}.csv'
    if expected_csv.exists():
        return expected_csv

    normalized_expected = normalize_text(expected_name)
    candidates = list(folder.glob('*.csv'))
    for path in candidates:
        if normalize_text(path.stem) == normalized_expected:
            return path

    raise FileNotFoundError(
        f'File for signal "{expected_name}" not found in {folder}. '
        f'Available csv: {[p.name for p in candidates]}'
    )


def normalize_text(text: str) -> str:
    return ''.join(ch for ch in text.lower() if ch.isalnum())


def detect_header_row(raw_df: pd.DataFrame) -> int:
    for idx, row in raw_df.iterrows():
        first = '' if pd.isna(row.iloc[0]) else str(row.iloc[0]).strip().lower()
        third = '' if len(row) < 3 or pd.isna(row.iloc[2]) else str(row.iloc[2]).strip().lower()
        if first == 'date/time' and third == 'value':
            return idx
        if first == 'date/time':
            return idx
    raise ValueError('Could not find the header row containing "Date/Time".')


def read_signal_csv(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, header=None, dtype=str, engine='python')
    if raw.empty:
        raise ValueError(f'{path} is empty.')

    header_row = detect_header_row(raw)
    data = raw.iloc[header_row + 1 :, :3].copy()
    data.columns = ['timestamp', 'unit', 'value']

    data['timestamp'] = data['timestamp'].astype(str).str.strip()
    data['value'] = data['value'].astype(str).str.replace(',', '.', regex=False).str.strip()

    data = data[(data['timestamp'] != '') & (data['timestamp'].str.lower() != 'nan')]
    data['timestamp'] = parse_datetime_series(data['timestamp'])
    data['value'] = pd.to_numeric(data['value'], errors='coerce')

    data = data.dropna(subset=['timestamp', 'value']).sort_values('timestamp').reset_index(drop=True)
    return data[['timestamp', 'value']]


def slice_session_window(df: pd.DataFrame, start_time: pd.Timestamp, duration_minutes: int) -> pd.DataFrame:
    end_time = start_time + pd.Timedelta(minutes=duration_minutes)
    mask = (df['timestamp'] >= start_time) & (df['timestamp'] < end_time)
    window = df.loc[mask].copy()
    if window.empty:
        return window
    window['elapsed_seconds'] = (window['timestamp'] - start_time).dt.total_seconds().astype(int)
    window['elapsed_minutes'] = window['elapsed_seconds'] / 60.0
    return window


def build_signal_dataset(session_infos: List[SessionInfo], signal_group: str, signal_name: str) -> pd.DataFrame:
    rows: List[pd.DataFrame] = []

    for session in session_infos:
        session_dir = RAW_ROOT / session.subject_id / session.condition
        file_path = find_signal_file(session_dir, signal_group, signal_name)
        raw_signal = read_signal_csv(file_path)
        window = slice_session_window(raw_signal, session.start_time, WINDOW_MINUTES)

        if window.empty:
            print(
                f'WARNING: empty extraction for subject={session.subject_id}, '
                f'condition={session.condition}, signal={signal_name}'
            )
            continue

        window['subject_id'] = session.subject_id
        window['condition'] = session.condition
        window['signal_group'] = signal_group
        window['signal_name'] = signal_name
        rows.append(window)

    if not rows:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    dataset = pd.concat(rows, ignore_index=True)
    dataset = dataset[OUTPUT_COLUMNS].sort_values(
        ['signal_name', 'condition', 'subject_id', 'timestamp']
    ).reset_index(drop=True)
    return dataset


def signal_output_name(signal_name: str) -> str:
    name = signal_name.lower().replace(' ', '_')
    return f'{name}.csv'


def main() -> None:
    session_infos = load_start_times(START_TIMES_PATH)
    PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

    summary_rows = []
    for signal_group, signal_names in SIGNAL_GROUPS.items():
        for signal_name in signal_names:
            dataset = build_signal_dataset(session_infos, signal_group, signal_name)
            out_path = PROCESSED_ROOT / signal_output_name(signal_name)
            save_csv(dataset, out_path)

            summary_rows.append({
                'signal_group': signal_group,
                'signal_name': signal_name,
                'output_file': str(out_path.relative_to(ROOT)),
                'n_rows': len(dataset),
                'n_subject_condition_pairs': dataset[['subject_id', 'condition']].drop_duplicates().shape[0] if not dataset.empty else 0,
                'min_timestamp': dataset['timestamp'].min() if not dataset.empty else pd.NaT,
                'max_timestamp': dataset['timestamp'].max() if not dataset.empty else pd.NaT,
            })
            print(f'Created {out_path.name}: {len(dataset)} rows')

    summary = pd.DataFrame(summary_rows)
    save_csv(summary, PROCESSED_ROOT / 'dataset_build_summary.csv')
    print(f'\nDone. Aggregated signal datasets are in {PROCESSED_ROOT}')


if __name__ == '__main__':
    main()